# TeachOnce Phase 0 — vLLM + NeMo Gym closed loop

1. Hit the OpenAI-compatible API on **Brev** through your Mac tunnel.
2. Sketch the **NeMo Gym → reward → NeMo-RL** closed loop (how we'd do RL).

**Prereq:** `brev port-forward teachonce-gpu -p 8000:8000` (or `./scripts/tunnel_vllm.sh`) in another terminal.

Uses repo `.env` — never points at third-party LLM hosts.

In [11]:
from __future__ import annotations

import json
import os
import urllib.error
import urllib.request
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "smoke-test":
    ROOT = ROOT.parent

env_path = ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

BASE = os.environ.get("NVIDIA_VLLM_BASE_URL", "http://127.0.0.1:8000/v1").rstrip("/")
KEY = os.environ.get("LOCAL_VLLM_API_KEY", "teachonce-local-key")
MODEL = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-1.5B-Instruct")

assert "openai.com" not in BASE.lower(), f"NVIDIA-only policy: refuse {BASE}"
print("BASE=", BASE)
print("MODEL=", MODEL)

BASE= http://127.0.0.1:8000/v1
MODEL= Qwen/Qwen2.5-1.5B-Instruct


## 1. List models

In [12]:
def api(method: str, path: str, body: dict | None = None) -> dict:
    url = f"{BASE}{path}"
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(
        url,
        data=data,
        method=method,
        headers={
            "Authorization": f"Bearer {KEY}",
            "Content-Type": "application/json",
        },
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            return json.loads(resp.read().decode())
    except urllib.error.URLError as e:
        raise RuntimeError(
            f"{method} {url} failed: {e}\n"
            "Is the tunnel up?  brev port-forward teachonce-gpu -p 8000:8000"
        ) from e

models = api("GET", "/models")
ids = [m["id"] for m in models.get("data", [])]
print("models:", ids)
assert MODEL in ids or any(MODEL in i for i in ids), f"expected {MODEL} in {ids}"
print("OK: /v1/models")

ConnectionResetError: [Errno 54] Connection reset by peer

## 2. Chat completion

In [ ]:
chat = api(
    "POST",
    "/chat/completions",
    {
        "model": MODEL,
        "messages": [{"role": "user", "content": "Reply with exactly: pong"}],
        "max_tokens": 16,
        "temperature": 0,
    },
)
text = chat["choices"][0]["message"]["content"]
print("assistant:", text)
print("usage:", chat.get("usage"))
assert text and text.strip(), "empty completion"
print("OK: /v1/chat/completions")

assistant: pong
usage: {'prompt_tokens': 34, 'total_tokens': 36, 'completion_tokens': 2, 'prompt_tokens_details': None}
OK: /v1/chat/completions


## 3. NeMo Gym closed loop (RL sketch)

Phase 0 already ran the **open half**: Gym rollouts + `verify()` rewards against local vLLM.

```text
┌─────────────┐   prompt / tools    ┌──────────────┐
│ NeMo Gym    │ ──────────────────► │ Agent        │
│ Resources   │◄── tool results ─── │ (mcqa_simple)│
│  + verify() │                     └──────┬───────┘
└──────▲──────┘                            │ chat/completions
       │ reward                            ▼
       │                          ┌────────────────┐
       │                          │ vLLM (policy)  │  ◄── Brev A100
       │                          └────────▲───────┘
       │                                   │ refit weights
┌──────┴───────────────────────────────────┴───────┐
│ NeMo-RL (GRPO) — closes the loop                 │
│  batch → Gym run_rollouts → (tokens, logp, r)    │
│       → advantage → policy update → refit vLLM   │
└──────────────────────────────────────────────────┘
```

| Piece | Phase 0 (done) | RL closed loop |
|-------|----------------|----------------|
| Env | `mcqa` resources server | same, or TeachOnce SOP env later |
| Policy | stock Qwen on vLLM | trainable policy; NeMo-RL owns vLLM HTTP |
| Score | `verify()` → `mean/reward` | same reward feeds GRPO |
| Update | none | GRPO / on-policy distill via NeMo-RL |

Gym is **CPU-only** (no GPU). Inference stays on vLLM; Gym proxies through its Model Server.

### 3a. One Gym eval iteration (what we already run on Brev)

This is **one outer RL step’s data collection** without the weight update:

In [ ]:
# Commands that already green on teachonce-gpu (Phase 0 smoke).
# Run on the VM — notebook only documents them.
GYM_EVAL_ONCE = """
# on Brev:
source ~/teachonce/.env && source ~/teachonce/gym-venv/bin/activate
export PATH="$HOME/.local/bin:$PATH"
# servers already up via scripts/start_vllm.sh + start_gym.sh
cd ~/teachonce && RUN_GYM=1 bash scripts/smoke_test.sh
# → results/mcqa_rollouts.jsonl
# → results/mcqa_rollouts_aggregate_metrics.json  (mean/reward, pass@1)
"""
print(GYM_EVAL_ONCE)

# Optional: pull metrics from Brev into this notebook kernel
import subprocess
import shutil

pull = ROOT / "smoke-test" / "_pulled_mcqa_metrics.json"
if shutil.which("brev") and not pull.exists():
    try:
        raw = subprocess.check_output(
            [
                "brev",
                "exec",
                "teachonce-gpu",
                "cat",
                "~/teachonce/results/mcqa_rollouts_aggregate_metrics.json",
            ],
            text=True,
            timeout=60,
        )
        # brev exec prefixes instance name lines — keep JSON object only
        start = raw.find("{")
        if start >= 0:
            pull.write_text(raw[start:])
            print("pulled", pull)
    except Exception as e:
        print("skip pull:", e)

if pull.exists():
    metrics = json.loads(pull.read_text())
    print("aggregate:", json.dumps(metrics, indent=2)[:800])
else:
    print("No local metrics file — run GYM_EVAL_ONCE on Brev or brev exec cat …")


# on Brev:
source ~/teachonce/.env && source ~/teachonce/gym-venv/bin/activate
export PATH="$HOME/.local/bin:$PATH"
# servers already up via scripts/start_vllm.sh + start_gym.sh
cd ~/teachonce && RUN_GYM=1 bash scripts/smoke_test.sh
# → results/mcqa_rollouts.jsonl
# → results/mcqa_rollouts_aggregate_metrics.json  (mean/reward, pass@1)

skip pull: Command '['brev', 'exec', 'teachonce-gpu', 'cat', '~/teachonce/results/mcqa_rollouts_aggregate_metrics.json']' timed out after 60 seconds
No local metrics file — run GYM_EVAL_ONCE on Brev or brev exec cat …


### 3b. Closed loop = Gym rollouts inside NeMo-RL GRPO

NeMo Gym does **not** update weights. **NeMo-RL** owns the outer loop and calls Gym each step:

1. **Refit** — sync latest policy weights into vLLM workers (`expose_http_server: true`).
2. **Rollout** — `NemoGym.run_rollouts(batch)` → Agent `POST /run` → model + tools → `verify()` → `(token_ids, logprobs, reward)`.
3. **Learn** — GRPO advantages from rewards → policy gradient update.
4. Goto 1.

Config shape (from [NeMo Gym ↔ NeMo-RL integration](https://docs.nvidia.com/nemo/rl/latest/design-docs/nemo-gym-integration.html)):

In [ ]:
# Illustrative NeMo-RL yaml + outer loop (not executed here).
# Real entrypoints live in NeMo-RL: examples/nemo_gym/run_grpo_nemo_gym.py

NEMO_RL_GYM_SNIPPET = """
policy:
  generation:
    backend: vllm
    vllm_cfg:
      async_engine: true
      expose_http_server: true   # Gym Model Server proxies here

env:
  should_use_nemo_gym: true
  nemo_gym:
    config_paths:
      - resources_servers/mcqa/configs/mcqa.yaml          # or TeachOnce env later
      - responses_api_agents/simple_agent/configs/simple_agent.yaml
"""

def rl_closed_loop_sketch(num_steps: int = 100):
    """What NeMo-RL does each training step (conceptual)."""
    for step in range(num_steps):
        # 1) policy weights → vLLM generation workers
        # refit_vllm(policy)

        # 2) Gym collects multi-turn rollouts + verify() rewards
        # batch = sample_tasks(train_jsonl)
        # rollouts = nemo_gym.run_rollouts(batch)
        #   each: prompt_token_ids, response_token_ids, logprobs, reward∈[0,1]

        # 3) GRPO update from group-relative advantages
        # loss = grpo_loss(rollouts)
        # policy.backward(loss); optimizer.step()

        # 4) (optional) eval: same gym eval run --limit N we used in Phase 0
        yield {"step": step, "note": "rollout→reward→update→refit"}

print(NEMO_RL_GYM_SNIPPET)
print("one step:", next(rl_closed_loop_sketch()))
print("docs: https://docs.nvidia.com/nemo/rl/latest/design-docs/nemo-gym-integration.html")


policy:
  generation:
    backend: vllm
    vllm_cfg:
      async_engine: true
      expose_http_server: true   # Gym Model Server proxies here

env:
  should_use_nemo_gym: true
  nemo_gym:
    config_paths:
      - resources_servers/mcqa/configs/mcqa.yaml          # or TeachOnce env later
      - responses_api_agents/simple_agent/configs/simple_agent.yaml

one step: {'step': 0, 'note': 'rollout→reward→update→refit'}
docs: https://docs.nvidia.com/nemo/rl/latest/design-docs/nemo-gym-integration.html


### 3c. TeachOnce mapping (later)

| Today (mcqa smoke) | Product RL |
|--------------------|------------|
| MCQ `example.jsonl` | SOP / Procedure Package tasks |
| `mcqa` `verify()` accuracy | DAG schema valid + executor pass / judge |
| Qwen 1.5B | Nemotron Nano 9B (+ LoRA) on Brev |
| `gym eval run` once | NeMo-RL GRPO with `should_use_nemo_gym: true` |

**NVIDIA-only:** `policy_base_url` / NeMo-RL vLLM HTTP must stay on our Brev endpoint — never `api.openai.com`.

Phase 0 gate stays: prove Gym↔vLLM rewards. Closing the loop = add NeMo-RL around that same Gym path.

In [ ]:
print("Phase 0 vLLM smoke OK.")
print(f"  endpoint: {BASE}")
print(f"  model:    {MODEL}")
print("NeMo Gym closed loop: eval/reward path proven on Brev;")
print("  RL update = NeMo-RL GRPO around the same Gym rollouts (section 3).")

Phase 0 vLLM smoke OK.
  endpoint: http://127.0.0.1:8000/v1
  model:    Qwen/Qwen2.5-1.5B-Instruct
NeMo Gym closed loop: eval/reward path proven on Brev;
  RL update = NeMo-RL GRPO around the same Gym rollouts (section 3).
